# Spare — more models (only if the test asks for them)
Same data as Q1/Q2 (cell 1). Each section is standalone after cell 1 has run.
A. Scratch: gradient descent · Ridge · KNN · your stump Boost · Boost with trees · log-target trick
B. sklearn: SVM · RandomForest · ExtraTrees · HistGB · AdaBoost · XGBoost · Voting · Stacking · GridSearch tuning

In [1]:
import os, warnings
os.environ["LOKY_MAX_CPU_COUNT"] = "4"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../titanic.csv")
cols = ["Pclass", "Sex", "Age", "SibSp", "Parch"]
df["Sex"] = df["Sex"].map({"male": 1, "female": 0})
df = df[cols + ["Survived", "Fare"]]
df = df.fillna(df.median()).sample(frac=1, random_state=42)
x, y, yc = df[cols].values.astype(float), df["Fare"].values, df["Survived"].values
n = int(len(df) * 0.8)
x_train, x_test, y_train, y_test = x[:n], x[n:], y[:n], y[n:]
yc_train, yc_test = yc[:n], yc[n:]

def r2(a, p):
    return 1 - ((a - p) ** 2).sum() / ((a - a.mean()) ** 2).sum()

## A1. Linear with gradient descent (if the teacher says 'use GD')

In [2]:
def linear_gd(x_train, y_train, x_test, lr=0.05, epochs=3000):
    mu, sd = x_train.mean(0), x_train.std(0)                        # scale first, or GD explodes
    a = np.column_stack([np.ones(len(x_train)), (x_train - mu) / sd])
    b = np.column_stack([np.ones(len(x_test)), (x_test - mu) / sd])
    w = np.zeros(a.shape[1])
    for _ in range(epochs):
        w -= lr * (2 / len(y_train)) * a.T @ (a @ w - y_train)
    return b @ w

print("GD R2:", round(r2(y_test, linear_gd(x_train, y_train, x_test)), 4))

GD R2: 0.5003


## A2. Ridge (linear with penalty — stops poly from going crazy)

In [3]:
def ridge(x_train, y_train, x_test, lam=10):
    mu, sd = x_train.mean(0), x_train.std(0)
    a = np.column_stack([np.ones(len(x_train)), (x_train - mu) / sd])
    b = np.column_stack([np.ones(len(x_test)), (x_test - mu) / sd])
    i = np.eye(a.shape[1]); i[0, 0] = 0                              # don't punish the bias
    w = np.linalg.inv(a.T @ a + lam * i) @ a.T @ y_train
    return b @ w

def poly2(x):                                                        # x, x^2 and every xi*xj
    out = [x]
    for i in range(x.shape[1]):
        for j in range(i, x.shape[1]):
            out.append(x[:, [i]] * x[:, [j]])
    return np.hstack(out)

for lam in [0.1, 1, 10, 100]:
    print("lam", lam, "poly2+ridge R2:", round(r2(y_test, ridge(poly2(x_train), y_train, poly2(x_test), lam)), 4))

lam 0.1 poly2+ridge R2: 0.6581
lam 1 poly2+ridge R2: 0.664
lam 10 poly2+ridge R2: 0.6374
lam 100 poly2+ridge R2: 0.5527


## A3. KNN regression (scratch)

In [4]:
def knn_reg(x_train, y_train, x_test, k=7):
    mu, sd = x_train.mean(0), x_train.std(0)
    a, b = (x_train - mu) / sd, (x_test - mu) / sd
    out = []
    for row in b:
        d = np.sqrt(((a - row) ** 2).sum(axis=1))
        near = np.argsort(d)[:k]
        out.append(y_train[near].mean())
    return np.array(out)

for k in [3, 7, 15]:
    print("k", k, "KNN R2:", round(r2(y_test, knn_reg(x_train, y_train, x_test, k)), 4))

k 3 KNN R2: 0.1989
k 7 KNN R2: 0.6238
k 15 KNN R2: 0.6483


## A4. Your Boost (stumps) and Boost with deeper trees

In [5]:
def boost(X_train, y_train, X_test, rounds=50, lr=0.1):          # your original
    pred_train = np.full(len(y_train), y_train.mean())
    pred_test = np.full(len(X_test), y_train.mean())
    for _ in range(rounds):
        error = y_train - pred_train
        best = 1e18
        for j in range(X_train.shape[1]):
            for t in np.unique(X_train[:, j])[:-1]:
                left = X_train[:, j] <= t
                a, b = error[left].mean(), error[~left].mean()
                fix_train = np.where(left, a, b)
                loss = ((error - fix_train) ** 2).sum()
                if loss < best:
                    best, best_train, best_test = loss, fix_train, np.where(X_test[:, j] <= t, a, b)
        pred_train += lr * best_train
        pred_test += lr * best_test
    return pred_test

print("Boost (stumps) R2:", round(r2(y_test, boost(x_train, y_train, x_test)), 4))

Boost (stumps) R2: 0.5344


In [6]:
# same tree code as Q1
def tree(x, y, depth):
    if depth == 0 or len(y) < 10:
        return y.mean()
    best_loss, best = ((y - y.mean()) ** 2).sum(), None
    for j in range(x.shape[1]):
        for t in np.unique(np.percentile(x[:, j], range(5, 100, 5))):
            left = x[:, j] <= t
            if left.sum() < 5 or (~left).sum() < 5:
                continue
            loss = ((y[left] - y[left].mean()) ** 2).sum() + ((y[~left] - y[~left].mean()) ** 2).sum()
            if loss < best_loss:
                best_loss, best = loss, (j, t)
    if best is None:
        return y.mean()
    j, t = best
    left = x[:, j] <= t
    return (j, t, tree(x[left], y[left], depth - 1), tree(x[~left], y[~left], depth - 1))

def tree_predict(node, x):
    if not isinstance(node, tuple):
        return np.full(len(x), node)
    j, t, l, r = node
    left = x[:, j] <= t
    out = np.empty(len(x))
    out[left], out[~left] = tree_predict(l, x[left]), tree_predict(r, x[~left])
    return out

def boost_trees(x_train, y_train, x_test, rounds=200, lr=0.05, depth=3):
    pred_train = np.full(len(y_train), y_train.mean())
    pred_test = np.full(len(x_test), y_train.mean())
    for _ in range(rounds):
        t = tree(x_train, y_train - pred_train, depth)          # tree learns the error left
        pred_train += lr * tree_predict(t, x_train)
        pred_test += lr * tree_predict(t, x_test)
    return pred_test

print("Boost (depth-3 trees) R2:", round(r2(y_test, boost_trees(x_train, y_train, x_test)), 4))

Boost (depth-3 trees) R2: 0.483


## A5. Log-target trick (prices / skewed y ≥ 0) and blending

In [7]:
p_log = np.expm1(boost_trees(x_train, np.log1p(y_train), x_test))   # learn log(y), turn back with expm1
print("Boost on log(y) R2:", round(r2(y_test, p_log), 4))

p_blend = (ridge(poly2(x_train), y_train, poly2(x_test), 10) + boost_trees(x_train, y_train, x_test)) / 2
print("Blend poly+boost R2:", round(r2(y_test, p_blend), 4))

Boost on log(y) R2: 0.6479


Blend poly+boost R2: 0.6243


## B. sklearn extra classifiers / regressors

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier,
                              RandomForestRegressor, GradientBoostingRegressor, StackingRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, r2_score

more = {
    "SVM": make_pipeline(StandardScaler(), SVC(C=3, random_state=42)),
    "RandomForest": RandomForestClassifier(300, random_state=42),
    "ExtraTrees": ExtraTreesClassifier(300, min_samples_leaf=2, random_state=42),
    "HistGB": HistGradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}
try:
    from xgboost import XGBClassifier
    more["XGBoost"] = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, verbosity=0)
except ImportError:
    print("no xgboost (pip install xgboost)")

for name, m in more.items():
    p = m.fit(x_train, yc_train).predict(x_test)
    print(f"{name:16s} acc {accuracy_score(yc_test, p):.4f}  F1 {f1_score(yc_test, p):.4f}")

SVM              acc 0.8156  F1 0.7317


RandomForest     acc 0.8212  F1 0.7500
ExtraTrees       acc 0.8324  F1 0.7541


HistGB           acc 0.8324  F1 0.7619
AdaBoost         acc 0.7933  F1 0.7338
GradientBoosting acc 0.8156  F1 0.7360


XGBoost          acc 0.8156  F1 0.7442


In [9]:
# Voting = average the probabilities of several models (often beats each one)
vote = VotingClassifier([("gb", GradientBoostingClassifier(random_state=42)),
                         ("rf", RandomForestClassifier(300, random_state=42)),
                         ("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))], voting="soft")
p = vote.fit(x_train, yc_train).predict(x_test)
print("Voting acc", round(accuracy_score(yc_test, p), 4), "F1", round(f1_score(yc_test, p), 4))

# Stacking for regression = a small model learns how to mix the others
stack = StackingRegressor([("gb", GradientBoostingRegressor(random_state=42)),
                           ("rf", RandomForestRegressor(300, random_state=42))], final_estimator=Ridge())
print("Stacking R2", round(r2_score(y_test, stack.fit(x_train, y_train).predict(x_test)), 4))

Voting acc 0.8324 F1 0.7619


Stacking R2 0.4392


## B2. Tuning with GridSearchCV (tries every combination with 5-fold CV)

In [10]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
                    {"n_estimators": [100, 300], "learning_rate": [0.05, 0.1], "max_depth": [2, 3, 4]},
                    cv=5, scoring="accuracy")
grid.fit(x_train, yc_train)
print("best params:", grid.best_params_, "| CV acc", round(grid.best_score_, 4))
p = grid.best_estimator_.predict(x_test)
print("test acc", round(accuracy_score(yc_test, p), 4))
# KNN: {"kneighborsclassifier__n_neighbors": [5, 11, 21]} on make_pipeline(StandardScaler(), KNeighborsClassifier())
# Tree: {"max_depth": [3, 4, 5, 6, None]}

best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100} | CV acc 0.8216
test acc 0.8324
